# Week 14: ML Fundamentals & Supervised Learning — Applied Statistics with Python (2026)

This week we enter the world of **machine learning (ML)** — algorithms that learn patterns from data to make predictions. We focus on the ML workflow, core concepts, and the most important supervised learning algorithms using **scikit-learn**.

| # | Topic |
|---|-------|
| 1 | What is Machine Learning? |
| 2 | The ML Workflow |
| 3 | Data Preprocessing & Feature Engineering |
| 4 | Train-Test Split & Cross-Validation |
| 5 | k-Nearest Neighbors (kNN) |
| 6 | Decision Trees |
| 7 | Random Forests |
| 8 | Logistic Regression |
| 9 | Support Vector Machines (SVM) |
| 10 | Model Evaluation Metrics |
| 11 | Practical Example: End-to-End ML Pipeline |
| 12 | Summary & Homework |

## Imports

We import scikit-learn for machine learning, along with our usual data and visualization libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Models
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, SVR

# Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, mean_squared_error, 
                             mean_absolute_error, r2_score)

# Datasets
from sklearn.datasets import load_iris, load_wine, make_classification, make_moons

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
np.random.seed(42)

import sklearn
print(f'scikit-learn version: {sklearn.__version__}')
print('All imports successful!')

---
## 1. What is Machine Learning?

**Machine Learning** is the field of study that gives computers the ability to learn from data without being explicitly programmed.

### Types of Machine Learning

| Type | Goal | Examples |
|------|------|----------|
| **Supervised** | Predict an output from labeled data | Classification, Regression |
| **Unsupervised** | Find patterns in unlabeled data | Clustering, Dimensionality Reduction |
| **Reinforcement** | Learn actions via rewards/penalties | Game AI, Robotics |

### Supervised Learning: Classification vs Regression

| Task | Target Variable | Examples | Metrics |
|------|----------------|----------|----------|
| **Classification** | Categorical (discrete) | Spam/Not Spam, Disease diagnosis | Accuracy, F1, AUC |
| **Regression** | Continuous (numeric) | House price, Temperature | MSE, R², MAE |

### Key Terminology

| Term | Meaning |
|------|---------|
| **Feature (X)** | Input variable used for prediction |
| **Target (y)** | Output variable to predict |
| **Training set** | Data used to learn the model |
| **Test set** | Data used to evaluate the model |
| **Overfitting** | Model memorizes training data, fails on new data |
| **Underfitting** | Model is too simple, misses patterns |
| **Hyperparameter** | Setting chosen before training (e.g., k in kNN) |

### 1.1 Statistics vs Machine Learning

Let's see how the same problem looks from both perspectives.

In [ ]:
# Generate sample data: exam score predicting pass/fail
np.random.seed(42)
n = 100
hours_studied = np.random.uniform(1, 10, n)
passed = (hours_studied + np.random.normal(0, 1.5, n) > 5).astype(int)

# Statistics approach: Logistic Regression with inference
import statsmodels.api as sm
X_stats = sm.add_constant(hours_studied)
logit_model = sm.Logit(passed, X_stats).fit(disp=0)
print('=== Statistics Approach (Inference) ===')
print(f'Coefficient: {logit_model.params[1]:.4f} (p={logit_model.pvalues[1]:.4f})')
print(f'Interpretation: Each hour increases log-odds of passing by {logit_model.params[1]:.3f}')
print(f'Focus: WHY does studying affect outcomes?\n')

# ML approach: Logistic Regression for prediction
X_ml = hours_studied.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X_ml, passed, 
                                                      test_size=0.2, random_state=42)
lr = LogisticRegression()
lr.fit(X_train, y_train)
accuracy = lr.score(X_test, y_test)
print('=== ML Approach (Prediction) ===')
print(f'Test accuracy: {accuracy:.1%}')
print(f'Prediction for 7 hours: {"Pass" if lr.predict([[7]])[0] else "Fail"}')
print(f'Focus: CAN we predict who passes?')

---
## 2. The ML Workflow

Every ML project follows a standard workflow:

1. **Define the problem** — What are we predicting? Classification or regression?
2. **Collect & explore data** — EDA, understand features
3. **Preprocess data** — Handle missing values, encode categories, scale features
4. **Split data** — Train/test (or cross-validation)
5. **Train model(s)** — Fit algorithms to training data
6. **Evaluate** — Measure performance on test data
7. **Tune hyperparameters** — Optimize model settings
8. **Deploy / report** — Use the model or present findings

### 2.1 The scikit-learn API

scikit-learn has a beautifully consistent API — all models follow the same pattern.

In [ ]:
# scikit-learn's consistent API: fit → predict → score
# Every model works the same way:

# 1. Load data
iris = load_iris()
X, y = iris.data, iris.target
print(f'Iris dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Features: {iris.feature_names}')
print(f'Classes: {iris.target_names}')

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

# 3. Create model → fit → predict → evaluate
model = KNeighborsClassifier(n_neighbors=5)  # create
model.fit(X_train, y_train)                   # fit (learn)
predictions = model.predict(X_test)            # predict
accuracy = model.score(X_test, y_test)         # evaluate

print(f'\nkNN (k=5) accuracy: {accuracy:.1%}')
print(f'\nThis same pattern works for ALL scikit-learn models!')

---
## 3. Data Preprocessing & Feature Engineering

Real-world data is messy. Before feeding data to ML models, we need to:

| Step | Why | Tool |
|------|-----|------|
| Handle missing values | Models can't process NaN | `SimpleImputer` |
| Encode categories | Models need numbers | `LabelEncoder`, `OneHotEncoder` |
| Scale features | Many models are distance-based | `StandardScaler`, `MinMaxScaler` |
| Engineer features | Create informative inputs | Domain knowledge |

### 3.1 Handling Missing Values

scikit-learn's `SimpleImputer` fills missing values with a strategy (mean, median, most frequent).

In [ ]:
# Create sample data with missing values
np.random.seed(42)
df_messy = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 35, np.nan, 28, 50, 42, 33],
    'income': [50000, np.nan, 75000, 90000, np.nan, 60000, 45000, np.nan, 85000, 70000],
    'education': ['BS', 'MS', 'BS', 'PhD', 'MS', 'BS', np.nan, 'PhD', 'MS', 'BS'],
    'purchased': [0, 1, 1, 1, 0, 0, 0, 1, 1, 0]
})
print('Messy data:')
print(df_messy)
print(f'\nMissing values:\n{df_messy.isna().sum()}')

# Impute numeric columns with median
num_imputer = SimpleImputer(strategy='median')
df_messy[['age', 'income']] = num_imputer.fit_transform(df_messy[['age', 'income']])

# Impute categorical columns with most frequent
cat_imputer = SimpleImputer(strategy='most_frequent')
df_messy[['education']] = cat_imputer.fit_transform(df_messy[['education']])

print('\nAfter imputation:')
print(df_messy)
print(f'\nMissing values: {df_messy.isna().sum().sum()}')

### 3.2 Encoding Categorical Variables

ML models need numeric inputs. We encode categories as numbers using Label Encoding or One-Hot Encoding.

In [ ]:
# Method 1: Label Encoding — assigns integers (good for ordinal data)
le = LabelEncoder()
df_messy['education_encoded'] = le.fit_transform(df_messy['education'])
print('Label Encoding:')
print(dict(zip(le.classes_, le.transform(le.classes_))))

# Method 2: One-Hot Encoding — creates binary columns (good for nominal data)
df_onehot = pd.get_dummies(df_messy[['education']], prefix='edu', drop_first=False)
print('\nOne-Hot Encoding:')
print(df_onehot)

# When to use which:
print('\n=== Encoding Guide ===')
print('Label Encoding: ordinal categories (Low < Medium < High)')
print('One-Hot Encoding: nominal categories (Red, Blue, Green — no order)')
print('Warning: Label encoding nominal data implies false ordering!')

### 3.3 Feature Scaling

Many algorithms (kNN, SVM, logistic regression) are sensitive to feature scales. A feature with range [0, 100000] will dominate one with range [0, 1].

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Sample data with very different scales
data = pd.DataFrame({
    'age': [25, 30, 45, 35, 50],
    'income': [50000, 75000, 90000, 60000, 120000],
    'score': [0.8, 0.6, 0.9, 0.7, 0.95]
})

print('Original data:')
print(data.describe().loc[['mean', 'std', 'min', 'max']])

# StandardScaler: mean=0, std=1 (most common)
scaler_std = StandardScaler()
data_std = pd.DataFrame(scaler_std.fit_transform(data), columns=data.columns)
print('\nStandardScaler (z-score normalization):')
print(data_std.describe().loc[['mean', 'std', 'min', 'max']].round(2))

# MinMaxScaler: range [0, 1]
scaler_mm = MinMaxScaler()
data_mm = pd.DataFrame(scaler_mm.fit_transform(data), columns=data.columns)
print('\nMinMaxScaler (range [0,1]):')
print(data_mm.describe().loc[['mean', 'std', 'min', 'max']].round(2))

print('\n=== Scaling Guide ===')
print('StandardScaler: most algorithms (kNN, SVM, Logistic Regression, PCA)')
print('MinMaxScaler: neural networks, algorithms needing bounded input')
print('No scaling needed: tree-based models (Decision Tree, Random Forest)')

---
## 4. Train-Test Split & Cross-Validation

We must evaluate models on **unseen data** to estimate real-world performance. Never evaluate on training data — it gives overly optimistic results.

### 4.1 Train-Test Split

The simplest approach: hold out a portion of data for testing.

In [ ]:
# Load the wine dataset for demonstration
wine = load_wine()
X, y = wine.data, wine.target
print(f'Wine dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes')

# Basic split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'\nBasic split:')
print(f'  Train: {len(y_train)} ({len(y_train)/len(y):.0%})')
print(f'  Test:  {len(y_test)} ({len(y_test)/len(y):.0%})')

# Stratified split — preserves class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nStratified split (class proportions preserved):')
for c in np.unique(y):
    train_pct = (y_train == c).mean()
    test_pct = (y_test == c).mean()
    print(f'  Class {c}: train={train_pct:.1%}, test={test_pct:.1%}')

### 4.2 Cross-Validation

A single train-test split can be lucky or unlucky. **k-Fold Cross-Validation** gives a more robust estimate by splitting the data k times.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 5-fold cross-validation
model = KNeighborsClassifier(n_neighbors=5)
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print('5-Fold Cross-Validation:')
print(f'  Fold scores: {scores.round(3)}')
print(f'  Mean: {scores.mean():.3f} ± {scores.std():.3f}')

# Compare with single train-test split
model.fit(X_train, y_train)
single_score = model.score(X_test, y_test)
print(f'\nSingle train-test split: {single_score:.3f}')
print(f'Cross-validation mean:   {scores.mean():.3f}')
print(f'\nCV gives a more reliable estimate of model performance.')

# Visualize CV scores
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 6), scores, color='steelblue', alpha=0.7)
ax.axhline(scores.mean(), color='red', ls='--', label=f'Mean = {scores.mean():.3f}')
ax.fill_between([0.5, 5.5], scores.mean()-scores.std(), scores.mean()+scores.std(),
                 alpha=0.1, color='red', label=f'±1 std = {scores.std():.3f}')
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('5-Fold Cross-Validation Results')
ax.legend()
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

---
## 5. k-Nearest Neighbors (kNN)

**kNN** is the simplest ML algorithm: to classify a new point, find the k closest training points and vote.

**How it works:**
1. Calculate distance from new point to all training points
2. Find the k nearest neighbors
3. For classification: majority vote among k neighbors
4. For regression: average value of k neighbors

**Key hyperparameter:** k (number of neighbors)
- Small k → complex boundary, risk of overfitting
- Large k → smooth boundary, risk of underfitting

### 5.1 kNN Classification with Iris

Let's apply kNN to the Iris dataset and visualize the decision boundary.

In [ ]:
# Use only 2 features for visualization
iris = load_iris()
X_2d = iris.data[:, [2, 3]]  # petal length, petal width
y_iris = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X_2d, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

# Scale features (important for kNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train kNN with different k values
k_values = [1, 3, 5, 15]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for idx, k in enumerate(k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    acc = knn.score(X_test_scaled, y_test)
    
    # Decision boundary
    x_min, x_max = X_train_scaled[:, 0].min()-1, X_train_scaled[:, 0].max()+1
    y_min, y_max = X_train_scaled[:, 1].min()-1, X_train_scaled[:, 1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='Set1')
    for c in range(3):
        mask = y_train == c
        axes[idx].scatter(X_train_scaled[mask, 0], X_train_scaled[mask, 1],
                          label=iris.target_names[c], s=20, alpha=0.7)
    axes[idx].set_title(f'k={k}, Acc={acc:.1%}', fontsize=11)
    if idx == 0:
        axes[idx].legend(fontsize=8)

plt.suptitle('kNN Decision Boundaries (Iris: Petal Features)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Finding the Optimal k

We use cross-validation to find the best k value.

In [ ]:
# Test k from 1 to 30
k_range = range(1, 31)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append({'k': k, 'mean': scores.mean(), 'std': scores.std()})

df_cv = pd.DataFrame(cv_scores)
best_k = df_cv.loc[df_cv['mean'].idxmax(), 'k']

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_cv['k'], df_cv['mean'], 'b-o', markersize=4)
ax.fill_between(df_cv['k'], df_cv['mean']-df_cv['std'], 
                 df_cv['mean']+df_cv['std'], alpha=0.15)
ax.axvline(best_k, color='red', ls='--', label=f'Best k={int(best_k)}')
ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('CV Accuracy')
ax.set_title('kNN: Choosing Optimal k')
ax.legend()
plt.tight_layout()
plt.show()

# Final model
best_knn = KNeighborsClassifier(n_neighbors=int(best_k))
best_knn.fit(X_train_scaled, y_train)
print(f'Best k = {int(best_k)}')
print(f'Test accuracy: {best_knn.score(X_test_scaled, y_test):.1%}')

---
## 6. Decision Trees

**Decision Trees** learn a hierarchy of if-else rules from the data. They recursively split features to create the most pure groups.

**Advantages:**
- Easy to interpret and visualize
- No feature scaling needed
- Handles both numeric and categorical data

**Disadvantages:**
- Prone to overfitting (deep trees memorize noise)
- Unstable (small data changes → different tree)

### 6.1 Training a Decision Tree

Let's train a decision tree on the Iris dataset and visualize the tree structure.

In [ ]:
# Train decision tree (no scaling needed)
X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train_dt, y_train_dt)

print(f'Training accuracy: {dt.score(X_train_dt, y_train_dt):.1%}')
print(f'Test accuracy:     {dt.score(X_test_dt, y_test_dt):.1%}')

# Visualize the tree
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(dt, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, ax=ax, fontsize=10)
ax.set_title('Decision Tree for Iris Classification', fontsize=14)
plt.tight_layout()
plt.show()

### 6.2 Overfitting in Decision Trees

Without depth limits, decision trees will perfectly fit the training data but fail on new data. Let's visualize this.

In [ ]:
# Compare different max_depth values
depths = range(1, 16)
train_scores = []
test_scores = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_dt, y_train_dt)
    train_scores.append(dt.score(X_train_dt, y_train_dt))
    test_scores.append(dt.score(X_test_dt, y_test_dt))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_scores, 'b-o', label='Training', markersize=5)
ax.plot(depths, test_scores, 'r-s', label='Test', markersize=5)
ax.set_xlabel('Tree Depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Overfitting with Increasing Depth')
ax.legend()
ax.set_ylim(0.7, 1.05)

# Mark the overfitting region
best_depth = depths[np.argmax(test_scores)]
ax.axvline(best_depth, color='green', ls='--', alpha=0.5, label=f'Best depth={best_depth}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Training accuracy keeps increasing → memorizing noise')
print(f'Test accuracy peaks at depth={best_depth}, then may decrease')
print(f'This gap is the hallmark of OVERFITTING.')

### 6.3 Feature Importance

Decision trees can tell us which features are most important for making predictions.

In [ ]:
# Train a tree and inspect feature importance
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_dt, y_train_dt)

importances = pd.Series(dt.feature_importances_, index=iris.feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Decision Tree Feature Importance')
plt.tight_layout()
plt.show()

print('Feature importance values:')
for name, imp in importances.items():
    print(f'  {name}: {imp:.4f}')
print(f'\nPetal features dominate — consistent with botanical knowledge!')

---
## 7. Random Forests

**Random Forests** solve the overfitting problem of decision trees by building many trees and averaging their predictions.

**How it works (Bagging + Feature Randomness):**
1. Create N bootstrap samples (random subsets with replacement)
2. Train a decision tree on each bootstrap sample
3. At each split, consider only a random subset of features
4. Final prediction: majority vote (classification) or average (regression)

| Hyperparameter | What it controls | Default |
|----------------|------------------|---------|
| `n_estimators` | Number of trees | 100 |
| `max_depth` | Maximum tree depth | None (unlimited) |
| `max_features` | Features per split | sqrt(n_features) |
| `min_samples_split` | Min samples to split a node | 2 |

### 7.1 Random Forest vs Single Decision Tree

Let's compare a Random Forest against a single decision tree.

In [ ]:
# Compare on the wine dataset
wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Single Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt_scores = cross_val_score(dt, X_train, y_train, cv=5)
dt.fit(X_train, y_train)
dt_test = dt.score(X_test, y_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_scores = cross_val_score(rf, X_train, y_train, cv=5)
rf.fit(X_train, y_train)
rf_test = rf.score(X_test, y_test)

print('=== Decision Tree vs Random Forest ===')
print(f'{"":<20} {"CV Mean":>10} {"CV Std":>10} {"Test":>10}')
print('-' * 55)
print(f'{"Decision Tree":<20} {dt_scores.mean():>10.3f} {dt_scores.std():>10.3f} {dt_test:>10.3f}')
print(f'{"Random Forest":<20} {rf_scores.mean():>10.3f} {rf_scores.std():>10.3f} {rf_test:>10.3f}')
print(f'\nRandom Forest: higher accuracy AND lower variance!')

### 7.2 Effect of Number of Trees

How many trees do we need? Let's see how accuracy improves as we add more trees.

In [ ]:
# Performance vs number of trees
n_trees_range = [1, 5, 10, 25, 50, 100, 200, 500]
results = []

for n in n_trees_range:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    scores = cross_val_score(rf, X_train, y_train, cv=5)
    rf.fit(X_train, y_train)
    results.append({
        'n_trees': n,
        'cv_mean': scores.mean(),
        'cv_std': scores.std(),
        'test': rf.score(X_test, y_test)
    })

df_trees = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_trees['n_trees'], df_trees['cv_mean'], 'b-o', label='CV Mean')
ax.fill_between(df_trees['n_trees'], 
                 df_trees['cv_mean'] - df_trees['cv_std'],
                 df_trees['cv_mean'] + df_trees['cv_std'], alpha=0.15)
ax.plot(df_trees['n_trees'], df_trees['test'], 'r--s', label='Test')
ax.set_xlabel('Number of Trees')
ax.set_ylabel('Accuracy')
ax.set_title('Random Forest: Accuracy vs Number of Trees')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
plt.show()

print('More trees generally helps but with diminishing returns.')
print('n_estimators=100 is usually a good starting point.')

### 7.3 Random Forest Feature Importance

Random Forests provide more stable feature importance than single trees.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=wine.feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('Random Forest Feature Importance (Wine Dataset)')
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
for name, imp in importances.nlargest(5).items():
    print(f'  {name}: {imp:.4f}')

---
## 8. Logistic Regression

Despite its name, **Logistic Regression** is a classification algorithm. It models the probability of belonging to a class using the logistic (sigmoid) function:

$$P(y=1|x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p)}}$$

**Key properties:**
- Linear decision boundary
- Outputs probabilities (not just predictions)
- Naturally extends to multiclass (one-vs-rest or softmax)
- Regularizable (L1/L2 penalty)

### 8.1 Logistic Regression on the Wine Dataset

Let's train logistic regression and examine the predicted probabilities.

In [ ]:
# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

# Predictions and probabilities
y_pred = lr.predict(X_test_s)
y_proba = lr.predict_proba(X_test_s)

print(f'Test accuracy: {lr.score(X_test_s, y_test):.1%}')
print(f'\nPredicted probabilities for first 5 test samples:')
prob_df = pd.DataFrame(y_proba[:5], columns=wine.target_names)
prob_df['Prediction'] = [wine.target_names[p] for p in y_pred[:5]]
prob_df['Actual'] = [wine.target_names[a] for a in y_test[:5]]
print(prob_df.round(3).to_string())

print('\nNote: Logistic regression gives probability estimates,')
print('which is useful for ranking and thresholding decisions.')

### 8.2 Regularization Effect

The `C` parameter controls regularization strength (smaller C = stronger regularization).

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
results = []

for C in C_values:
    lr = LogisticRegression(C=C, max_iter=1000, random_state=42)
    scores = cross_val_score(lr, X_train_s, y_train, cv=5)
    lr.fit(X_train_s, y_train)
    results.append({
        'C': C,
        'cv_mean': scores.mean(),
        'test': lr.score(X_test_s, y_test),
        'n_nonzero': np.sum(np.abs(lr.coef_) > 0.01)
    })

df_reg = pd.DataFrame(results)
print('Logistic Regression: Effect of Regularization (C)')
print(df_reg.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(df_reg['C'], df_reg['cv_mean'], 'b-o', label='CV')
ax.semilogx(df_reg['C'], df_reg['test'], 'r--s', label='Test')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('Accuracy')
ax.set_title('Logistic Regression: Regularization Effect')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Support Vector Machines (SVM)

**SVM** finds the optimal hyperplane that maximally separates classes. It's particularly powerful for:
- High-dimensional data
- Non-linear boundaries (via the **kernel trick**)

| Kernel | Boundary | When to use |
|--------|----------|-------------|
| `linear` | Straight line/hyperplane | Linearly separable data |
| `rbf` (default) | Flexible curves | Most general-purpose |
| `poly` | Polynomial curves | Known polynomial relationship |

### 9.1 SVM with Different Kernels

Let's visualize how different kernels create different decision boundaries on the make_moons dataset.

In [ ]:
# Generate non-linear data (moons)
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_moons_s = scaler.fit_transform(X_moons)

kernels = ['linear', 'rbf', 'poly']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, kernel in enumerate(kernels):
    svm = SVC(kernel=kernel, random_state=42)
    svm.fit(X_moons_s, y_moons)
    acc = svm.score(X_moons_s, y_moons)
    
    # Decision boundary
    x_min, x_max = X_moons_s[:, 0].min()-0.5, X_moons_s[:, 0].max()+0.5
    y_min, y_max = X_moons_s[:, 1].min()-0.5, X_moons_s[:, 1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    axes[idx].scatter(X_moons_s[y_moons==0, 0], X_moons_s[y_moons==0, 1],
                       c='blue', s=15, alpha=0.6, label='Class 0')
    axes[idx].scatter(X_moons_s[y_moons==1, 0], X_moons_s[y_moons==1, 1],
                       c='red', s=15, alpha=0.6, label='Class 1')
    axes[idx].set_title(f'SVM kernel={kernel}\nAcc={acc:.1%}', fontsize=11)

plt.suptitle('SVM: Different Kernels on Non-Linear Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Linear kernel fails on non-linear data.')
print('RBF kernel adapts to complex boundaries.')

### 9.2 SVM on the Wine Dataset

Let's apply SVM with hyperparameter tuning using GridSearchCV.

In [ ]:
# Grid search for best SVM parameters
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

svm = SVC(random_state=42)
grid_search = GridSearchCV(svm, param_grid, cv=5, scoring='accuracy', 
                           return_train_score=True)
grid_search.fit(X_train_s, y_train)

print('=== SVM GridSearchCV Results ===')
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.3f}')
print(f'Test accuracy: {grid_search.score(X_test_s, y_test):.3f}')

# Show top 5 parameter combinations
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df.nsmallest(5, 'rank_test_score')[[
    'param_C', 'param_kernel', 'param_gamma', 'mean_test_score', 'std_test_score'
]]
print('\nTop 5 configurations:')
print(top5.to_string(index=False))

---
## 10. Model Evaluation Metrics

Accuracy alone is not enough! Different metrics capture different aspects of model performance.

### Classification Metrics

| Metric | Formula | Best for |
|--------|---------|----------|
| **Accuracy** | (TP+TN) / Total | Balanced classes |
| **Precision** | TP / (TP+FP) | Minimizing false positives |
| **Recall** | TP / (TP+FN) | Minimizing false negatives |
| **F1 Score** | 2 × (Prec × Rec) / (Prec + Rec) | Imbalanced classes |
| **AUC-ROC** | Area under ROC curve | Comparing classifiers |

### 10.1 Confusion Matrix

The confusion matrix shows the full picture of classification performance.

In [ ]:
# Use the best model from SVM grid search
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_s)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=wine.target_names, yticklabels=wine.target_names)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized (percentages)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=axes[1],
            xticklabels=wine.target_names, yticklabels=wine.target_names)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.show()

# Detailed classification report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=wine.target_names))

### 10.2 ROC Curve and AUC

The **ROC curve** shows the trade-off between true positive rate and false positive rate. Higher AUC = better classifier.

In [ ]:
# Binary classification example for ROC
# Create binary problem: class 0 vs rest
y_binary_test = (y_test == 0).astype(int)

# Compare models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'kNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

fig, ax = plt.subplots(figsize=(8, 6))

for name, model in models.items():
    # Train on binary version
    y_binary_train = (y_train == 0).astype(int)
    
    if name == 'kNN (k=5)':
        model.fit(X_train_s, y_binary_train)
        y_score = model.predict_proba(X_test_s)[:, 1]
    else:
        model.fit(X_train_s, y_binary_train)
        y_score = model.predict_proba(X_test_s)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_binary_test, y_score)
    auc = roc_auc_score(y_binary_test, y_score)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')

# Diagonal (random classifier)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: Model Comparison', fontsize=14)
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

### 10.3 Model Comparison Summary

Let's compare all our models on the wine dataset in one comprehensive table.

In [ ]:
# Compare all models
all_models = {
    'kNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
}

comparison = []
for name, model in all_models.items():
    # Use scaled data for all (doesn't hurt tree-based models)
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=5)
    model.fit(X_train_s, y_train)
    test_acc = model.score(X_test_s, y_test)
    y_pred = model.predict(X_test_s)
    
    comparison.append({
        'Model': name,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Acc': test_acc,
        'F1 (weighted)': f1_score(y_test, y_pred, average='weighted')
    })

df_comp = pd.DataFrame(comparison).sort_values('Test Acc', ascending=False)
print('=== Model Comparison on Wine Dataset ===')
print(df_comp.to_string(index=False, float_format='{:.3f}'.format))

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = range(len(df_comp))
ax.bar(x_pos, df_comp['CV Mean'], yerr=df_comp['CV Std'], 
       capsize=5, color='steelblue', alpha=0.7, label='CV Score')
ax.scatter(x_pos, df_comp['Test Acc'], color='red', s=100, 
           zorder=5, label='Test Score', marker='D')
ax.set_xticks(x_pos)
ax.set_xticklabels(df_comp['Model'], rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison: Wine Classification')
ax.legend()
ax.set_ylim(0.8, 1.05)
plt.tight_layout()
plt.show()

---
## 11. Practical Example: End-to-End ML Pipeline

Let's build a complete ML pipeline for a realistic classification problem — predicting whether a customer will churn (leave) based on their behavior.

### Step 1: Create a Realistic Dataset

In [ ]:
# Generate synthetic customer churn data
np.random.seed(42)
n_customers = 1000

df_churn = pd.DataFrame({
    'tenure_months': np.random.randint(1, 72, n_customers),
    'monthly_charges': np.random.uniform(20, 120, n_customers).round(2),
    'total_charges': np.zeros(n_customers),  # will compute
    'num_products': np.random.choice([1, 2, 3, 4], n_customers, p=[0.4, 0.3, 0.2, 0.1]),
    'has_contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], 
                                      n_customers, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(['Credit card', 'Bank transfer', 'Electronic check'],
                                        n_customers, p=[0.4, 0.3, 0.3]),
    'num_support_calls': np.random.poisson(2, n_customers),
    'satisfaction_score': np.random.uniform(1, 5, n_customers).round(1),
})

# Compute total charges
df_churn['total_charges'] = (df_churn['tenure_months'] * 
                              df_churn['monthly_charges']).round(2)

# Generate churn based on realistic factors
churn_prob = (
    0.3 
    - 0.003 * df_churn['tenure_months']          # longer tenure → less churn
    + 0.002 * df_churn['monthly_charges']          # higher charges → more churn
    - 0.05 * df_churn['num_products']              # more products → less churn
    + 0.03 * df_churn['num_support_calls']         # more complaints → more churn
    - 0.08 * df_churn['satisfaction_score']         # higher satisfaction → less churn
    + 0.15 * (df_churn['has_contract'] == 'Month-to-month')  # no contract → more churn
    + np.random.normal(0, 0.05, n_customers)
)
churn_prob = np.clip(churn_prob, 0.05, 0.95)
df_churn['churned'] = np.random.binomial(1, churn_prob)

# Add some missing values (realistic)
mask = np.random.random(n_customers) < 0.05
df_churn.loc[mask, 'satisfaction_score'] = np.nan

print(f'Dataset shape: {df_churn.shape}')
print(f'Churn rate: {df_churn["churned"].mean():.1%}')
print(f'Missing values: {df_churn.isna().sum().sum()}')
print(f'\nFirst 5 rows:')
print(df_churn.head())

### Step 2: Exploratory Data Analysis

Before modeling, let's understand the data and its relationship with churn.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Churn by contract type
ct = pd.crosstab(df_churn['has_contract'], df_churn['churned'], normalize='index')
ct.plot(kind='bar', stacked=True, ax=axes[0,0], color=['steelblue', 'coral'])
axes[0,0].set_title('Churn by Contract Type')
axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=0)
axes[0,0].legend(['Stayed', 'Churned'])

# 2. Tenure distribution
df_churn.groupby('churned')['tenure_months'].hist(
    ax=axes[0,1], alpha=0.6, bins=20, label=['Stayed', 'Churned'])
axes[0,1].set_title('Tenure Distribution')
axes[0,1].legend(['Stayed', 'Churned'])

# 3. Monthly charges
df_churn.boxplot(column='monthly_charges', by='churned', ax=axes[0,2])
axes[0,2].set_title('Monthly Charges by Churn')
axes[0,2].set_xlabel('Churned')
plt.sca(axes[0,2])
plt.title('Monthly Charges by Churn')

# 4. Support calls
df_churn.groupby('churned')['num_support_calls'].mean().plot(
    kind='bar', ax=axes[1,0], color=['steelblue', 'coral'])
axes[1,0].set_title('Avg Support Calls')
axes[1,0].set_xticklabels(['Stayed', 'Churned'], rotation=0)

# 5. Satisfaction score
df_churn.boxplot(column='satisfaction_score', by='churned', ax=axes[1,1])
axes[1,1].set_title('Satisfaction by Churn')
plt.sca(axes[1,1])
plt.title('Satisfaction by Churn')

# 6. Correlation heatmap (numeric columns only)
numeric_cols = df_churn.select_dtypes(include=[np.number]).columns
corr = df_churn[numeric_cols].corr()
sns.heatmap(corr[['churned']].sort_values('churned'), annot=True, 
            cmap='RdBu_r', center=0, ax=axes[1,2], fmt='.2f')
axes[1,2].set_title('Correlation with Churn')

plt.suptitle('Customer Churn EDA', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Step 3: Build the ML Pipeline

Using scikit-learn Pipelines to chain preprocessing and modeling steps — this prevents data leakage and makes the workflow reproducible.

In [ ]:
# Define features and target
feature_cols = ['tenure_months', 'monthly_charges', 'total_charges',
                'num_products', 'num_support_calls', 'satisfaction_score',
                'has_contract', 'payment_method']

X = df_churn[feature_cols].copy()
y = df_churn['churned'].copy()

# Identify numeric and categorical columns
numeric_features = ['tenure_months', 'monthly_charges', 'total_charges',
                     'num_products', 'num_support_calls', 'satisfaction_score']
categorical_features = ['has_contract', 'payment_method']

# Preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # fill missing values
    ('scaler', StandardScaler())                     # standardize
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))  # one-hot encode
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Churn rate — Train: {y_train.mean():.1%}, Test: {y_test.mean():.1%}')

### Step 4: Train and Compare Multiple Models

Let's train several models using the same pipeline and compare their performance.

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'kNN (k=7)': KNeighborsClassifier(n_neighbors=7),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
}

results = []
trained_pipelines = {}

for name, model in models.items():
    # Create full pipeline: preprocess → model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Cross-validation
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
    
    # Fit on full training set and evaluate on test
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'CV F1 Mean': cv_scores.mean(),
        'CV F1 Std': cv_scores.std(),
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred),
        'Test Recall': recall_score(y_test, y_pred),
        'Test F1': f1_score(y_test, y_pred),
        'Test AUC': roc_auc_score(y_test, y_proba)
    })
    trained_pipelines[name] = pipeline

df_results = pd.DataFrame(results).sort_values('Test F1', ascending=False)
print('=== Model Comparison (Customer Churn) ===')
print(df_results.to_string(index=False, float_format='{:.3f}'.format))

### Step 5: Detailed Analysis of Best Model

Let's pick the best model and analyze it in detail.

In [ ]:
# Best model based on F1
best_name = df_results.iloc[0]['Model']
best_pipeline = trained_pipelines[best_name]
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print(f'Best model: {best_name}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))

# Visualization dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=['Stayed', 'Churned'], yticklabels=['Stayed', 'Churned'])
axes[0,0].set_xlabel('Predicted')
axes[0,0].set_ylabel('Actual')
axes[0,0].set_title('Confusion Matrix')

# 2. ROC Curve
for name, pipeline in trained_pipelines.items():
    y_p = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_p)
    auc = roc_auc_score(y_test, y_p)
    axes[0,1].plot(fpr, tpr, lw=2, label=f'{name} ({auc:.3f})')
axes[0,1].plot([0,1], [0,1], 'k--')
axes[0,1].set_title('ROC Curves')
axes[0,1].set_xlabel('FPR')
axes[0,1].set_ylabel('TPR')
axes[0,1].legend(fontsize=8)

# 3. Prediction probability distribution
axes[1,0].hist(y_proba[y_test==0], bins=30, alpha=0.6, label='Stayed', color='steelblue')
axes[1,0].hist(y_proba[y_test==1], bins=30, alpha=0.6, label='Churned', color='coral')
axes[1,0].axvline(0.5, color='black', ls='--')
axes[1,0].set_title('Predicted Churn Probability Distribution')
axes[1,0].set_xlabel('P(Churn)')
axes[1,0].legend()

# 4. Feature importance (if available)
if hasattr(best_pipeline.named_steps['classifier'], 'feature_importances_'):
    # Get feature names after preprocessing
    cat_names = best_pipeline.named_steps['preprocessor'].named_transformers_[
        'cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
    all_features = numeric_features + list(cat_names)
    importances = pd.Series(
        best_pipeline.named_steps['classifier'].feature_importances_,
        index=all_features
    ).sort_values()
    importances.plot(kind='barh', ax=axes[1,1], color='teal')
    axes[1,1].set_title('Feature Importance')
elif hasattr(best_pipeline.named_steps['classifier'], 'coef_'):
    cat_names = best_pipeline.named_steps['preprocessor'].named_transformers_[
        'cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
    all_features = numeric_features + list(cat_names)
    coefs = pd.Series(
        np.abs(best_pipeline.named_steps['classifier'].coef_[0]),
        index=all_features
    ).sort_values()
    coefs.plot(kind='barh', ax=axes[1,1], color='teal')
    axes[1,1].set_title('Feature Coefficients (|β|)')
else:
    axes[1,1].text(0.5, 0.5, 'Feature importance\nnot available', 
                    ha='center', va='center', fontsize=14)
    axes[1,1].set_title('Feature Importance')

plt.suptitle(f'Churn Prediction — Best Model: {best_name}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 6: Business Recommendations

Let's translate model insights into actionable business recommendations.

In [ ]:
# Identify high-risk customers in the test set
test_results = X_test.copy()
test_results['churn_prob'] = y_proba
test_results['actual_churned'] = y_test.values
test_results['predicted_churn'] = y_pred

# High-risk customers (top 20% by churn probability)
high_risk = test_results.nlargest(int(len(test_results) * 0.2), 'churn_prob')

print('=== BUSINESS RECOMMENDATIONS ===')
print(f'\nTotal test customers: {len(test_results)}')
print(f'High-risk customers (top 20%): {len(high_risk)}')
print(f'Actual churn rate in high-risk group: {high_risk["actual_churned"].mean():.1%}')
print(f'Actual churn rate overall: {y_test.mean():.1%}')
print(f'\nThe model concentrates {high_risk["actual_churned"].sum()} of '
      f'{y_test.sum()} churners ({high_risk["actual_churned"].sum()/y_test.sum():.0%}) '
      f'in the top 20% of predictions.')

# Profile of high-risk customers
print(f'\nHigh-risk customer profile (vs all customers):')
for col in ['tenure_months', 'monthly_charges', 'num_support_calls', 'satisfaction_score']:
    hr_mean = high_risk[col].mean()
    all_mean = test_results[col].mean()
    direction = 'higher' if hr_mean > all_mean else 'lower'
    print(f'  {col}: {hr_mean:.1f} ({direction} than avg {all_mean:.1f})')

print(f'\nContract type in high-risk group:')
print(high_risk['has_contract'].value_counts(normalize=True).to_string())

print(f'\nACTION ITEMS:')
print('1. Target high-risk customers with retention offers')
print('2. Encourage month-to-month customers to switch to contracts')
print('3. Proactively reach out to customers with many support calls')
print('4. Offer loyalty benefits to newer customers (low tenure)')

---
## 12. Summary

| Section | Key Concepts | Python Tools |
|---------|-------------|---------------|
| 1. What is ML | Supervised/unsupervised, classification/regression | Conceptual |
| 2. ML Workflow | fit → predict → score, consistent API | `sklearn` API |
| 3. Preprocessing | Missing values, encoding, scaling | `SimpleImputer`, `OneHotEncoder`, `StandardScaler` |
| 4. Train-Test & CV | Stratified split, k-fold cross-validation | `train_test_split`, `cross_val_score` |
| 5. kNN | Distance-based, choosing k | `KNeighborsClassifier` |
| 6. Decision Trees | Rule-based splits, overfitting, feature importance | `DecisionTreeClassifier`, `plot_tree` |
| 7. Random Forests | Ensemble of trees, bagging, n_estimators | `RandomForestClassifier` |
| 8. Logistic Regression | Probability output, regularization (C) | `LogisticRegression` |
| 9. SVM | Kernels (linear, rbf), GridSearchCV | `SVC`, `GridSearchCV` |
| 10. Evaluation | Confusion matrix, precision/recall/F1, ROC/AUC | `classification_report`, `roc_curve` |
| 11. ML Pipeline | ColumnTransformer, Pipeline, end-to-end workflow | `Pipeline`, `ColumnTransformer` |

## Homework

### Task 1: Iris Classification Pipeline
Build a complete scikit-learn Pipeline for the Iris dataset:
1. Use `StandardScaler` + `KNeighborsClassifier` in a Pipeline.
2. Use `GridSearchCV` to find the best k from 1 to 20.
3. Report the best k, CV score, and test accuracy.
4. Plot the confusion matrix and classification report.

### Task 2: Decision Tree vs Random Forest
Using the wine dataset:
1. Train a Decision Tree with max_depth values from 1 to 15.
2. Train Random Forests with n_estimators in [10, 50, 100, 200].
3. Plot training vs test accuracy for both models.
4. Which model generalizes better? At what settings?

### Task 3: Model Comparison on a New Dataset
Use `sklearn.datasets.load_breast_cancer()` (binary classification):
1. Preprocess the data (scale features).
2. Train at least 4 different classifiers.
3. Compare using 5-fold CV with F1 score.
4. Plot ROC curves for all models on one figure.
5. Which model would you recommend and why?

### Task 4: Regression Challenge
Use `sklearn.datasets.fetch_california_housing()` (regression):
1. Train `KNeighborsRegressor`, `DecisionTreeRegressor`, and `RandomForestRegressor`.
2. Evaluate using RMSE, MAE, and R².
3. Tune the best model using GridSearchCV.
4. Plot actual vs predicted values for the best model.

---
## Next Week Preview

**Week 15: Unsupervised Learning & Model Evaluation** — We'll explore algorithms that find structure in unlabeled data: k-Means clustering, hierarchical clustering, PCA for dimensionality reduction, and advanced model evaluation techniques.

---
*Applied Statistics with Python (2026) | Week 14 | ML Fundamentals & Supervised Learning*